# 07 · Statistical analysis
**Step 9.** For each problem: hypothesis → test → assumptions → p-value → decision → business insight.

| Test | Question |
|---|---|
| Independent two-sample t-test | Do delayed orders receive significantly lower review scores than on-time orders? |
| One-way ANOVA | Does order value differ significantly across product categories? |
| Chi-square test of independence | Is payment method associated with order status? |

Implementation: `src/stats_tests.py`. α = 0.05.

In [ ]:
import sys, warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from src import config
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
sns.set_theme(style="whitegrid", palette="viridis")

# Where the raw CSVs live. To rehearse on generated test data instead, use:
#   RAW_DIR = config.SYNTHETIC_DIR   (run `python scripts/make_synthetic_data.py` first)
RAW_DIR = config.RAW_DIR

In [ ]:
from src.db import get_engine
from sqlalchemy import text

engine = get_engine()
print("Connected:", engine.url.render_as_string(hide_password=True))


def query(sql, params=None):
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn, params=params)

In [ ]:
from src.reporting import load_queries_module, default_params
from src.stats_tests import ttest_delivery_vs_review, anova_category_vs_order_value, chi_square_payment_vs_status

q = load_queries_module()
params = default_params(lambda s, p=None: query(s, p), q)
orders = query(q.QUERIES["stats_orders"], q.used_params(q.QUERIES["stats_orders"], params))
print(orders.shape)

## 1 · T-test: delivery delay vs review score

In [ ]:
t = ttest_delivery_vs_review(orders)
import json; print(json.dumps({k: v for k, v in t.items() if k != "assumptions"}, indent=2, default=str))
display(pd.json_normalize(t["assumptions"]).T)
d = orders.dropna(subset=["delivery_status", "review_score"])
sns.barplot(data=d, x="delivery_status", y="review_score", errorbar=("ci", 95)); plt.ylim(0, 5)
plt.title("Mean review score (95% CI)"); plt.show()
print("Business insight:", t["business_insight"])

## 2 · ANOVA: category vs order value

In [ ]:
a = anova_category_vs_order_value(orders)
print(json.dumps({k: v for k, v in a.items() if k not in ("assumptions", "category_mean_order_value")}, indent=2, default=str))
display(pd.json_normalize(a["assumptions"]).T)
sub = orders[(orders.is_valid_sale == 1) & orders.primary_category.isin(a["categories_tested"])]
sns.boxplot(data=sub, y="primary_category", x=np.log1p(sub["total_order_value"]), order=a["categories_tested"])
plt.xlabel("log(1 + order value)"); plt.title("Order value by category"); plt.show()
print("Business insight:", a["business_insight"])

## 3 · Chi-square: payment method vs order status

In [ ]:
c = chi_square_payment_vs_status(orders)
print(json.dumps({k: v for k, v in c.items() if k not in ("assumptions", "contingency_table")}, indent=2, default=str))
display(pd.DataFrame(c["contingency_table"]))
display(pd.json_normalize(c["assumptions"]).T)
print("Business insight:", c["business_insight"])